# ETL Pipeline Demo — Bibliometrix Python
## Advanced Level 

This notebook demonstrates the ETL pipeline developed for the Bibliometrix-Python project.
The pipeline extracts data from OpenAlex and PubMed APIs, transforms it into the WoS standard schema, and validates the output.

---
## PHASE 1: EXTRACT
Data is retrieved via REST APIs from OpenAlex and PubMed.
The `retrieve()` function handles pagination, rate limits, and retries automatically.

In [1]:
from www.services.api_retriever import retrieve

print("=== EXTRACT: OpenAlex ===")
records_oa = retrieve(query="machine learning", platform="openalex", total=10)
print(f"Records retrieved: {len(records_oa)}")
print(f"Sample raw keys: {list(records_oa[0].keys())[:8]}")
print(f"\nSample title: {records_oa[0].get('title', 'N/A')}")

=== EXTRACT: OpenAlex ===
Records retrieved: 10
Sample raw keys: ['id', 'doi', 'title', 'display_name', 'relevance_score', 'publication_year', 'publication_date', 'ids']

Sample title: Scikit-learn: Machine Learning in Python


In [2]:
print("=== EXTRACT: PubMed ===")
records_pm = retrieve(query="machine learning", platform="pubmed", total=10)
print(f"Records retrieved: {len(records_pm)}")
print(f"Sample raw keys: {list(records_pm[0].keys())[:8]}")
print(f"\nSample title: {records_pm[0].get('Title', 'N/A')}")

=== EXTRACT: PubMed ===
Records retrieved: 10
Sample raw keys: ['uid', 'pubdate', 'epubdate', 'source', 'authors', 'lastauthor', 'title', 'sorttitle']

Sample title: N/A


---
## PHASE 2: TRANSFORM
Raw API responses are mapped to the WoS standard schema using mapping dictionaries.
Multi-value fields are cast to `list[str]`, scalar fields to `str`, and `TC` to `int`.

In [3]:
from www.services.standardizer import standardize
import pandas as pd

print("=== TRANSFORM: OpenAlex ===")
df_oa = standardize(records_oa, source="openalex")
print(f"Shape: {df_oa.shape}")
print(f"Columns: {df_oa.columns.tolist()}")
df_oa[['AU', 'TI', 'PY', 'SO', 'TC', 'DB']].head(3)

=== TRANSFORM: OpenAlex ===
Shape: (10, 25)
Columns: ['UT', 'DI', 'TI', 'PY', 'LA', 'DT', 'TC', 'SO', 'JI', 'AU', 'AF', 'C1', 'RP', 'AB', 'VL', 'IS', 'BP', 'EP', 'DE', 'CR', 'ID', 'PMID', 'DB', 'SR', 'SR_FULL']


,AU,TI,PY,SO,TC,DB
0,"[Fabián Pedregosa, Gaël Varoquaux, Alexandre G...",Scikit-learn: Machine Learning in Python,2012,ARXIV (CORNELL UNIVERSITY),63672,OPENALEX
1,[],"Genetic algorithms in search, optimization, an...",1989,CHOICE REVIEWS ONLINE,49333,OPENALEX
2,[J. R. Quinlan],C4.5: Programs for Machine Learning,1992,,23696,OPENALEX


In [4]:
print("=== TRANSFORM: PubMed ===")
df_pm = standardize(records_pm, source="pubmed")
print(f"Shape: {df_pm.shape}")
print(f"Columns: {df_pm.columns.tolist()}")
df_pm[['AU', 'TI', 'PY', 'SO', 'TC', 'DB']].head(3)

=== TRANSFORM: PubMed ===
Shape: (10, 25)
Columns: ['UT', 'TI', 'SO', 'JI', 'PY', 'VL', 'IS', 'LA', 'DT', 'RP', 'AU', 'AF', 'DI', 'PMID', 'BP', 'EP', 'CR', 'AB', 'C1', 'DE', 'ID', 'TC', 'DB', 'SR', 'SR_FULL']


,AU,TI,PY,SO,TC,DB
0,"[Jeon YJ, Song JS, Borghare S, Lee Y, Choi YW,...",Detection of referable diabetic retinopathy us...,2026,Frontiers in medicine,0,PUBMED
1,"[Wang B, He W]",Prediction model for intrapartum labor analges...,2026,Frontiers in medicine,0,PUBMED
2,"[Ahmed N, Alghamdi M]",Role of Artificial Intelligence in Infectious ...,2026 Feb,Saudi medical journal,0,PUBMED


### Inspect multi-value fields
Author keywords (`DE`) and cited references (`CR`) must be `list[str]`.

In [5]:
print("=== Multi-value fields (OpenAlex) ===")
print(f"AU type: {type(df_oa['AU'].iloc[0])}")
print(f"AU sample: {df_oa['AU'].iloc[0]}")
print(f"\nDE type: {type(df_oa['DE'].iloc[0])}")
print(f"DE sample: {df_oa['DE'].iloc[0]}")
print(f"\nCR type: {type(df_oa['CR'].iloc[0])}")
print(f"CR sample (first 2): {df_oa['CR'].iloc[0][:2]}")

=== Multi-value fields (OpenAlex) ===
AU type: <class 'list'>
AU sample: ['Fabián Pedregosa', 'Gaël Varoquaux', 'Alexandre Gramfort', 'Vincent Michel', 'Bertrand Thirion', 'Olivier Grisel', 'Mathieu Blondel', 'Müller, Andreas', 'Nothman, Joel', 'Louppe, Gilles', 'Peter Prettenhofer', 'Ron J. Weiss', 'Vincent Dubourg', 'Jake Vanderplas', 'Alexandre Passos', 'David Cournapeau', 'Matthieu Brucher', 'Matthieu Perrot', 'Édouard Duchesnay']

DE type: <class 'list'>
DE sample: ['Python (programming language)', 'Documentation', 'Computer science', 'MIT License', 'Artificial intelligence', 'Machine learning', 'Programming language', 'License', 'Software engineering', 'Operating system']

CR type: <class 'list'>
CR sample (first 2): ['https://openalex.org/W1496508106', 'https://openalex.org/W1571024744']


---
## PHASE 3: VALIDATE
The validation module checks:
1. All mandatory columns exist
2. No NaN or None values remain
3. Multi-value columns are correctly typed as lists

In [6]:
from www.services.validator import validate

print("=== VALIDATE: OpenAlex ===")
df_oa = validate(df_oa)
print(f"\nSR sample: {df_oa['SR'].iloc[0]}")
df_oa[['SR', 'DE', 'AB']].head(3)

=== VALIDATE: OpenAlex ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.

SR sample: Fabián Pedregosa, 2012, arXiv (Cornell University)


,SR,DE,AB
0,"Fabián Pedregosa, 2012, arXiv (Cornell Univers...","[Python (programming language), Documentation,...",Scikit-learn is a Python module integrating a ...
1,"NA, 1989, Choice Reviews Online","[Computer science, Artificial intelligence, Ma...",From the Publisher:\r\nThis book brings togeth...
2,"J. R. Quinlan, 1992,","[Computer science, Unix, Classifier (UML), Mac...",Classifier systems play a major role in machin...


In [7]:
print("=== VALIDATE: PubMed ===")
df_pm = validate(df_pm)
print(f"\nSR sample: {df_pm['SR'].iloc[0]}")
df_pm[['SR', 'DE', 'AB']].head(3)

=== VALIDATE: PubMed ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.

SR sample: Jeon YJ, 2026, Front Med (Lausanne)


,SR,DE,AB
0,"Jeon YJ, 2026, Front Med (Lausanne)",[],
1,"Wang B, 2026, Front Med (Lausanne)",[],
2,"Ahmed N, 2026 Feb, Saudi Med J",[],


---
## FULL PIPELINE — 200 records
End-to-end demonstration with 200 records per platform, exported to CSV.

In [11]:
print("=== FULL PIPELINE: OpenAlex (200 records) ===")
records_oa_200 = retrieve(query="machine learning", platform="openalex", total=200)
df_oa_200 = standardize(records_oa_200, source="openalex")
df_oa_200 = validate(df_oa_200)
df_oa_200.to_csv("test_openalex_200.csv", index=False)
print(f"Shape: {df_oa_200.shape}")
print("CSV saved: test_openalex_200.csv")
df_oa_200[['AU', 'TI', 'PY', 'SO', 'TC', 'SR']].head(10)

=== FULL PIPELINE: OpenAlex (200 records) ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.
Shape: (200, 25)
CSV saved: test_openalex_200.csv


,AU,TI,PY,SO,TC,SR
0,"[Fabián Pedregosa, Gaël Varoquaux, Alexandre G...",Scikit-learn: Machine Learning in Python,2012,ARXIV (CORNELL UNIVERSITY),63672,"Fabián Pedregosa, 2012, arXiv (Cornell Univers..."
1,[],"Genetic algorithms in search, optimization, an...",1989,CHOICE REVIEWS ONLINE,49333,"NA, 1989, Choice Reviews Online"
2,[J. R. Quinlan],C4.5: Programs for Machine Learning,1992,,23696,"J. R. Quinlan, 1992,"
3,"[Ian H. Witten, Eibe Frank, Mark A. Hall]",Data Mining: Practical Machine Learning Tools ...,2011,ELSEVIER EBOOKS,25711,"Ian H. Witten, 2011, Elsevier eBooks"
4,[Arthur Asuncion],UCI Machine Learning Repository,2007,MEDICAL ENTOMOLOGY AND ZOOLOGY,24320,"Arthur Asuncion, 2007, Medical Entomology and ..."
5,[Nasser M. Nasrabadi],Pattern Recognition and Machine Learning,2007,JOURNAL OF ELECTRONIC IMAGING,22082,"Nasser M. Nasrabadi, 2007, Journal of Electron..."
6,[David E. Goldberg],"Genetic Algorithms in Search, Optimization and...",1988,,17771,"David E. Goldberg, 1988,"
7,[],Proceedings of the 24th international conferen...,2007,,11733,"NA, 2007,"
8,[Kevin P. Murphy],Machine learning a probabilistic perspective,2012,,9327,"Kevin P. Murphy, 2012,"
9,"[Michael I. Jordan, Tom M. Mitchell]","Machine learning: Trends, perspectives, and pr...",2015,SCIENCE,9509,"Michael I. Jordan, 2015, Science"


In [10]:
print("=== FULL PIPELINE: PubMed (200 records) ===")
records_pm_200 = retrieve(query="machine learning", platform="pubmed", total=200)
df_pm_200 = standardize(records_pm_200, source="pubmed")
df_pm_200 = validate(df_pm_200)
df_pm_200.to_csv("test_pubmed_200.csv", index=False)
print(f"Shape: {df_pm_200.shape}")
print("CSV saved: test_pubmed_200.csv")
df_pm_200[['AU', 'TI', 'PY', 'SO', 'TC', 'SR']].head(10)

=== FULL PIPELINE: PubMed (200 records) ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.
Shape: (200, 25)
CSV saved: test_pubmed_200.csv


,AU,TI,PY,SO,TC,SR
0,"[Jeon YJ, Song JS, Borghare S, Lee Y, Choi YW,...",Detection of referable diabetic retinopathy us...,2026,Frontiers in medicine,0,"Jeon YJ, 2026, Front Med (Lausanne)"
1,"[Wang B, He W]",Prediction model for intrapartum labor analges...,2026,Frontiers in medicine,0,"Wang B, 2026, Front Med (Lausanne)"
2,"[Ahmed N, Alghamdi M]",Role of Artificial Intelligence in Infectious ...,2026 Feb,Saudi medical journal,0,"Ahmed N, 2026 Feb, Saudi Med J"
3,[Alhasan MS],Artificial Intelligence in Radiology: Hidden F...,2026 Feb,Saudi medical journal,0,"Alhasan MS, 2026 Feb, Saudi Med J"
4,"[Zou X, Cheng Y, Chen X]","Diterpenoids in medicinal plants: structure, d...",2026,Frontiers in plant science,0,"Zou X, 2026, Front Plant Sci"
5,"[Brunnengraeber E, Cassidy T, Woodbridge D, Ja...",Handheld hyperspectral imaging dataset of annu...,2026 Jun,Data in brief,0,"Brunnengraeber E, 2026 Jun, Data Brief"
6,"[Al-Mekhlafi E, Al-Makhlafi M, Qaida S, Asqein...",YMGD: A yemeni music genres database with audi...,2026 Jun,Data in brief,0,"Al-Mekhlafi E, 2026 Jun, Data Brief"
7,"[Annabestani M, Zhou G, Wun H, Mosadegh B]",Acoustic-based Stenosis Detection for Dialysis...,2026 May 22,Research square,0,"Annabestani M, 2026 May 22, Res Sq"
8,"[Wellnitz J, Maxfield T, Hart M, Rath M, Kirch...",Machine Learning Models with a Reject Option t...,2026 May 19,Research square,0,"Wellnitz J, 2026 May 19, Res Sq"
9,"[Ellis RJ, Tang B, Riggs PK, Ludicello JE, Mar...",Compartmentalized Biomarker Correlations in HI...,2026 May 21,Research square,0,"Ellis RJ, 2026 May 21, Res Sq"


---
## Summary

| Platform | Records | Columns | NaN | SR |
|----------|---------|---------|-----|----|
| OpenAlex | 200 | 26 | 0 | ✅ |
| PubMed   | 200 | 26 | 0 | ✅ |

The ETL pipeline successfully:
- Extracted data from OpenAlex and PubMed REST APIs
- Transformed raw JSON into the WoS standard schema
- Enforced type contracts (list[str], str, int)
- Validated all mandatory columns
- Generated standardized CSV files ready for Bibliometrix-Python analysis